# 01 — Finance Methodology

**Goal:** validate market-data retrieval and the deterministic risk calculations before any agent/LangGraph code is written. Nothing about the multi-agent app gets built until the numbers here are proven correct.

## Outline

- **Step 0 — Ground truth methodology.** Decide how each function gets validated: hand-calculated toy examples (small price series checkable by calculator) *and* a cross-check against a trusted reference implementation — not just checking our code against itself.
- **Step 1 — Setup & sample portfolio.** Load `.env`, define the 5-stock portfolio (AAPL, NVDA, MSFT, GOOGL, TSLA) matching the `HOLDING` schema shape (`symbol`, `quantity`, `avg_cost`).
- **Step 2 — Fetch historical daily prices.** `get_historical_prices(symbol)` via Alpha Vantage `TIME_SERIES_DAILY`, with an offline fallback dataset so we're not blocked while waiting on API access.
- **Step 3 — `calculate_returns()`.** Daily % returns from a price series. Validate by hand on 2-3 known prices.
- **Step 4 — `calculate_volatility()`.** Annualized std dev of returns. Validate against a manual calculation and a reference implementation.
- **Step 5 — `calculate_max_drawdown()`.** Largest peak-to-trough decline. Validate on a synthetic price series where the answer is obvious by inspection.
- **Step 6 — `calculate_concentration()`.** Portfolio-level, weight-based (largest holding's % of total portfolio value). Needs current prices × quantities across all 5 holdings.
- **Step 7 — `calculate_sector_exposure()`.** Groups holdings by sector (from Alpha Vantage company overview) and sums weights per sector.
- **Step 8 — `calculate_correlation_matrix()`.** Pairwise return correlation across the 5 holdings.
- **Step 9 — `calculate_contribution_to_loss()`.** Each holding's share of total portfolio loss over the period — this later ranks News targets in the full-investigation route.
- **Step 10 — Full pipeline run.** Execute all seven calculations together across the real portfolio for a chosen date range, assemble the combined `risk_results` JSON exactly as it would be persisted to `INVESTIGATION.risk_results`, and sanity-check the numbers agree with each other as a whole.

Each step: explanation → code → validation, before moving to the next.

## Step 0 — Ground truth methodology

Before writing any risk function, we need to agree on what "validated against ground truth" actually means here. Two checks, used together for every function below:

1. **Hand-calculated toy example.** A tiny price series (4-5 points) small enough to compute the expected answer with a calculator, written out as plain arithmetic in the markdown so it's checkable by eye.
2. **Independent reference implementation.** A second, deliberately different code path for the same formula (e.g. plain-Python loop vs. numpy/pandas vectorized), asserted to agree with the "production" version. This catches bugs a single implementation checked only against its own toy example would miss — if both independently-written versions agree with the hand-calculated answer, we have much higher confidence.

No external finance library (e.g. `empyrical`) — keeping dependencies lean matches the project's "kept deliberately small" philosophy, and these formulas are simple enough that a from-scratch reference implementation is just as trustworthy.

The helper below does the tolerance-based comparison used throughout the notebook.

In [12]:
def assert_close(actual, expected, tol=1e-4, label=""):
    """Raise with a clear message if actual/expected differ by more than tol."""
    diff = abs(actual - expected)
    assert diff <= tol, f"{label}: expected {expected}, got {actual} (diff={diff})"
    print(f"OK  {label}: {actual:.6f}  (expected {expected:.6f})")


# --- Conventions used throughout this notebook (and later, the Risk Agent) ---
# Simple returns, not log returns: (P_t - P_{t-1}) / P_{t-1}.
#   Chosen because portfolio-level metrics (contribution-to-loss, concentration)
#   need returns/values that sum sensibly across holdings, and the eventual
#   report speaks in plain-English terms ("NVDA fell 14%") that simple returns
#   map to directly.
TRADING_DAYS_PER_YEAR = 252  # standard annualization factor for volatility

# All metrics are stored as decimals (0.284), not percentages (28.4) --
# matches the risk_results example in CLAUDE.md. Percent formatting only
# happens later, at the Answer/Report Agent synthesis step.

## Step 1 — Setup & sample portfolio

Load the Alpha Vantage API key from a local `.env` file (never committed). Define the 5-stock portfolio matching the `HOLDING` schema shape (`symbol`, `quantity`, `avg_cost`).

In [13]:
import time

import numpy as np
import pandas as pd

from risklensaidev.risk import ALPHA_VANTAGE_API_KEY, HAS_API_KEY, PORTFOLIO, SYMBOLS

print(f"Alpha Vantage API key loaded: {HAS_API_KEY}")
SYMBOLS

Alpha Vantage API key loaded: True


['AAPL', 'NVDA', 'MSFT', 'GOOGL', 'TSLA']

## Step 2 — Fetch historical daily prices

`get_historical_prices(symbol)` via Alpha Vantage `TIME_SERIES_DAILY`, with an offline fallback dataset so we're not blocked while waiting on real API access. This is the Market Agent's job in the real system — raw prices only, no derived returns.

` get_historical_prices `

Purpose:

Gets the past daily prices of a stock.

Explanation:

Risk calculations need historical prices. This is essentially the future Market Agent's data-fetching job.

*(Implementation, plus its `_fallback_prices()` helper, now lives in `src/risklensaidev/risk.py` — imported below rather than redefined here, so notebook 2 uses the exact same fetcher.)*

In [14]:
from risklensaidev.risk import get_historical_prices

In [15]:
# Fetch prices for every holding. Free-tier Alpha Vantage is rate-limited
# (5 requests/minute) -- pace real calls accordingly; no delay needed for fallback data.
prices = {}
for symbol in SYMBOLS:
    prices[symbol] = get_historical_prices(symbol)
    if HAS_API_KEY:
        time.sleep(12)

for symbol, df in prices.items():
    print(f"{symbol}: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

prices["NVDA"].tail()

AAPL: 100 rows, 2026-03-18 to 2026-08-10
NVDA: 100 rows, 2026-03-18 to 2026-08-10
MSFT: 100 rows, 2026-03-18 to 2026-08-10
GOOGL: 100 rows, 2026-03-18 to 2026-08-10
TSLA: 100 rows, 2026-03-18 to 2026-08-10


,open,high,low,close,volume
2026-08-04,211.30,213.0600,209.05,211.94,134921997.0
2026-08-05,216.86,222.2200,216.40,219.22,158187403.0
2026-08-06,221.53,223.6300,217.00,218.99,113940644.0
2026-08-07,221.54,224.7600,220.66,223.96,105669440.0
2026-08-10,223.40,224.1379,216.77,217.55,115846596.0


## Step 3 — `calculate_returns()`

Daily % returns from a price series. Validated by hand on 2-3 known prices, plus an independent reference implementation (see Step 0 methodology).

`calculate_returns() `

Purpose:

Calculates how much a stock went up or down each day.

Explanation:
Converts prices like $100 → $105 into a return like +5%. Other risk calculations need returns rather than just prices.


In [16]:
from risklensaidev.risk import calculate_returns

` _calculate_returns_reference `

Purpose:

Calculates daily returns a second way.

Explanation:
To test and check that calculate_returns() isn't giving the wrong answer.


In [17]:
def _calculate_returns_reference(close_prices: pd.Series) -> list:
    """Independent plain-Python reimplementation, used only to cross-check
    calculate_returns() above (see Step 0 methodology)."""
    values = close_prices.tolist()
    return [(values[i] - values[i - 1]) / values[i - 1] for i in range(1, len(values))]



In [18]:
# --- Validation: hand-calculated toy example ---
# Prices: 100 -> 110 -> 99 -> 108.9
#   day2: (110 - 100) / 100   =  0.10
#   day3: (99  - 110) / 110   = -0.10
#   day4: (108.9 - 99) / 99   =  0.10
toy_prices = pd.Series([100.0, 110.0, 99.0, 108.9])
toy_returns = calculate_returns(toy_prices)

expected = [0.10, -0.10, 0.10]
for i, exp in enumerate(expected):
    assert_close(toy_returns.iloc[i], exp, label=f"calculate_returns toy day{i + 2}")

# --- Validation: cross-check against the independent reference implementation ---
reference_returns = _calculate_returns_reference(toy_prices)
for i, ref in enumerate(reference_returns):
    assert_close(toy_returns.iloc[i], ref, label=f"calculate_returns vs reference day{i + 2}")

# Apply to the real portfolio
returns = {symbol: calculate_returns(df["close"]) for symbol, df in prices.items()}
returns["NVDA"].tail()

OK  calculate_returns toy day2: 0.100000  (expected 0.100000)
OK  calculate_returns toy day3: -0.100000  (expected -0.100000)
OK  calculate_returns toy day4: 0.100000  (expected 0.100000)
OK  calculate_returns vs reference day2: 0.100000  (expected 0.100000)
OK  calculate_returns vs reference day3: -0.100000  (expected -0.100000)
OK  calculate_returns vs reference day4: 0.100000  (expected 0.100000)


2026-08-04    0.025648
2026-08-05    0.034349
2026-08-06   -0.001049
2026-08-07    0.022695
2026-08-10   -0.028621
Name: close, dtype: float64

## Step 4 — `calculate_volatility()`

Annualized standard deviation of returns. Validated against a manual calculation and an independent reference implementation.

` calculate_volatility `

Purpose:
Measures how much a stock's daily returns move around.

Explanation:
To help answer how unstable/risky has this stock been? Higher volatility = bigger fluctuations.

In [19]:
from risklensaidev.risk import calculate_volatility

` _calculate_volatility_reference `

Purpose:
Calculates volatility manually a second way.

Explanation:
To test used to verify `calculate_volatility`


In [20]:

def _calculate_volatility_reference(returns_list: list) -> float:
    """Independent plain-Python reimplementation, used only to cross-check
    calculate_volatility() above (see Step 0 methodology)."""
    n = len(returns_list)
    mean = sum(returns_list) / n
    sample_variance = sum((r - mean) ** 2 for r in returns_list) / (n - 1)
    daily_std = sample_variance ** 0.5
    return daily_std * (TRADING_DAYS_PER_YEAR ** 0.5)



In [21]:


# --- Validation: hand-calculated toy example ---
# Reuse the Step 3 toy returns: [0.10, -0.10, 0.10]
#   mean            = 0.10/3                     = 0.033333
#   squared devs    = 0.004444, 0.017778, 0.004444
#   sample variance = sum / (n - 1) = 0.026667/2  = 0.013333
#   daily std       = sqrt(0.013333)              = 0.115470
#   annualized      = 0.115470 * sqrt(252)         ~ 1.83303
expected_daily_std = 0.1154700538
expected_annualized_vol = expected_daily_std * np.sqrt(TRADING_DAYS_PER_YEAR)

toy_volatility = calculate_volatility(toy_returns)
assert_close(toy_volatility, expected_annualized_vol, tol=1e-3, label="calculate_volatility toy example")

# --- Validation: cross-check against the independent reference implementation ---
reference_volatility = _calculate_volatility_reference(toy_returns.tolist())
assert_close(toy_volatility, reference_volatility, label="calculate_volatility vs reference")

# Apply to the real portfolio
volatility = {symbol: calculate_volatility(r) for symbol, r in returns.items()}
volatility


OK  calculate_volatility toy example: 1.833030  (expected 1.833030)
OK  calculate_volatility vs reference: 1.833030  (expected 1.833030)


{'AAPL': np.float64(0.289508270516201),
 'NVDA': np.float64(0.4024795570055936),
 'MSFT': np.float64(0.4105267463540114),
 'GOOGL': np.float64(0.3895716755771673),
 'TSLA': np.float64(0.5197756947724551)}

## Step 5 — `calculate_max_drawdown()`

Largest peak-to-trough decline. Validated on a synthetic price series where the answer is obvious by inspection.

` calculate_max_drawdown() `

Purpose:
Finds the worst fall from a previous high.

Explanation:
To help answer What was the worst decline this stock experienced during this period?

In [22]:
from risklensaidev.risk import calculate_max_drawdown

` _calculate_max_drawdown_reference `

Purpose:
Calculates the worst decline another way.

Explanation:
To Check the real drawdown function


In [23]:

def _calculate_max_drawdown_reference(prices_list: list) -> float:
    """Independent plain-Python reimplementation, used only to cross-check
    calculate_max_drawdown() above (see Step 0 methodology)."""
    peak = prices_list[0]
    max_dd = 0.0
    for p in prices_list:
        if p > peak:
            peak = p
        dd = (p - peak) / peak
        if dd < max_dd:
            max_dd = dd
    return max_dd



In [24]:

# --- Validation: hand-calculated toy example ---
# Prices: 100 -> 120 -> 90 -> 95 -> 130 -> 80 -> 110
# Tracking the running peak and drawdown at each point:
#   100 (peak=100, dd=0), 120 (peak=120, dd=0), 90 (peak=120, dd=-0.2500)
#   95  (peak=120, dd=-0.2083), 130 (peak=130, dd=0), 80 (peak=130, dd=-0.3846)
#   110 (peak=130, dd=-0.1538)
# The worst drawdown is obvious by inspection: peak 130 down to trough 80.
#   max_drawdown = (80 - 130) / 130 = -50/130 = -0.384615...
toy_dd_prices = pd.Series([100.0, 120.0, 90.0, 95.0, 130.0, 80.0, 110.0])
expected_max_drawdown = -50 / 130

toy_max_dd = calculate_max_drawdown(toy_dd_prices)
assert_close(toy_max_dd, expected_max_drawdown, label="calculate_max_drawdown toy example")

# --- Validation: cross-check against the independent reference implementation ---
reference_max_dd = _calculate_max_drawdown_reference(toy_dd_prices.tolist())
assert_close(toy_max_dd, reference_max_dd, label="calculate_max_drawdown vs reference")

# Apply to the real portfolio
max_drawdown = {symbol: calculate_max_drawdown(df["close"]) for symbol, df in prices.items()}
max_drawdown

OK  calculate_max_drawdown toy example: -0.384615  (expected -0.384615)
OK  calculate_max_drawdown vs reference: -0.384615  (expected -0.384615)


{'AAPL': np.float64(-0.1270621827411168),
 'NVDA': np.float64(-0.19398489861712062),
 'MSFT': np.float64(-0.2338443498653696),
 'GOOGL': np.float64(-0.21094332124583975),
 'TSLA': np.float64(-0.3300244795292744)}

## Step 6 — `calculate_concentration()`

Portfolio-level, weight-based (largest holding's % of total portfolio value). Needs current prices × quantities across all 5 holdings.

` calculate_concentration() `

Purpose:
Finds how much of your portfolio is invested in each company and which company is biggest.


Explanation:
Helps answer to answer Am I relying too heavily on one stock?

In [25]:
from risklensaidev.risk import calculate_concentration

` _calculate_concentration_reference`

Purpose: Calculates sector exposure another way.

Explanation:

To check the real sector-exposure calculation.

In [26]:

def _calculate_concentration_reference(portfolio: list, current_prices: dict) -> dict:
    """Independent plain-Python reimplementation (loop instead of dict
    comprehension), used only to cross-check calculate_concentration() above."""
    market_values = {}
    for h in portfolio:
        market_values[h["symbol"]] = h["quantity"] * current_prices[h["symbol"]]
    total = 0.0
    for mv in market_values.values():
        total += mv
    weights = {}
    for symbol, mv in market_values.items():
        weights[symbol] = mv / total
    return weights

In [27]:





# --- Validation: hand-calculated toy example ---
# 3 holdings, quantities 10 / 5 / 2, prices 10 / 20 / 100:
#   A: 10 * 10  = 100
#   B: 5  * 20  = 100
#   C: 2  * 100 = 200
#   total = 400 -> weights: A=0.25, B=0.25, C=0.50 (C is the largest holding)
toy_portfolio = [
    {"symbol": "A", "quantity": 10},
    {"symbol": "B", "quantity": 5},
    {"symbol": "C", "quantity": 2},
]
toy_current_prices = {"A": 10, "B": 20, "C": 100}

toy_concentration = calculate_concentration(toy_portfolio, toy_current_prices)
assert_close(toy_concentration["weights"]["A"], 0.25, label="concentration toy weight A")
assert_close(toy_concentration["weights"]["B"], 0.25, label="concentration toy weight B")
assert_close(toy_concentration["weights"]["C"], 0.50, label="concentration toy weight C")
assert toy_concentration["largest_holding_symbol"] == "C", "expected C to be the largest holding"
print("OK  largest_holding_symbol == C")

# --- Validation: cross-check against the independent reference implementation ---
reference_weights = _calculate_concentration_reference(toy_portfolio, toy_current_prices)
for symbol in reference_weights:
    assert_close(toy_concentration["weights"][symbol], reference_weights[symbol], label=f"concentration vs reference {symbol}")

# Apply to the real portfolio, using each symbol's latest close price
latest_prices = {symbol: df["close"].iloc[-1] for symbol, df in prices.items()}
concentration = calculate_concentration(PORTFOLIO, latest_prices)
concentration

OK  concentration toy weight A: 0.250000  (expected 0.250000)
OK  concentration toy weight B: 0.250000  (expected 0.250000)
OK  concentration toy weight C: 0.500000  (expected 0.500000)
OK  largest_holding_symbol == C
OK  concentration vs reference A: 0.250000  (expected 0.250000)
OK  concentration vs reference B: 0.250000  (expected 0.250000)
OK  concentration vs reference C: 0.500000  (expected 0.500000)


{'weights': {'AAPL': np.float64(0.18492068015009133),
  'NVDA': np.float64(0.1957576102964334),
  'MSFT': np.float64(0.24286241327906372),
  'GOOGL': np.float64(0.25736524323854143),
  'TSLA': np.float64(0.11909405303587016)},
 'largest_holding_symbol': 'GOOGL',
 'largest_holding_weight': np.float64(0.25736524323854143)}

## Step 7 — `calculate_sector_exposure()`

Groups holdings by sector (from Alpha Vantage company overview) and sums weights per sector.

`get_company_sector `

Purpose:
Finds what sector a company belongs to.

Explanation:
Needed so you can determine how much of the portfolio is Technology, Consumer Discretionary, etc.

*(Implementation, plus its `_fallback_sector()` helper, now lives in `src/risklensaidev/risk.py`.)*

In [28]:
from risklensaidev.risk import get_company_sector

`calculate_sector_exposure `

Purpose:
Calculates how much of your portfolio belongs to each sector.

Explanation:
Helps to answer "Am I too exposed to one industry/sector?"

In [29]:
from risklensaidev.risk import calculate_sector_exposure

`_calculate_sector_exposure_reference` 

Purpose:
Calculates sector exposure another way

Explanation:
To check the real sector-exposure calculation.

In [30]:
def _calculate_sector_exposure_reference(portfolio: list, current_prices: dict, sectors: dict) -> dict:
    """Independent reimplementation -- recomputes weights from scratch rather
    than reusing calculate_concentration(), used only to cross-check
    calculate_sector_exposure() above."""
    market_values = [h["quantity"] * current_prices[h["symbol"]] for h in portfolio]
    total = sum(market_values)
    exposure = {}
    for h, mv in zip(portfolio, market_values):
        sector = sectors[h["symbol"]]
        exposure[sector] = exposure.get(sector, 0.0) + mv / total
    return exposure



In [31]:


# --- Validation: hand-calculated toy example ---
# Reuse the Step 6 toy portfolio/prices (A=0.25, B=0.25, C=0.50 weight).
# Assign sectors: A -> Technology, B -> Technology, C -> Healthcare.
#   Technology exposure = 0.25 + 0.25 = 0.50
#   Healthcare exposure = 0.50
toy_sectors = {"A": "Technology", "B": "Technology", "C": "Healthcare"}

toy_sector_exposure = calculate_sector_exposure(toy_portfolio, toy_current_prices, toy_sectors)
assert_close(toy_sector_exposure["Technology"], 0.50, label="sector_exposure toy Technology")
assert_close(toy_sector_exposure["Healthcare"], 0.50, label="sector_exposure toy Healthcare")

# --- Validation: cross-check against the independent reference implementation ---
reference_exposure = _calculate_sector_exposure_reference(toy_portfolio, toy_current_prices, toy_sectors)
for sector in reference_exposure:
    assert_close(toy_sector_exposure[sector], reference_exposure[sector], label=f"sector_exposure vs reference {sector}")

# Apply to the real portfolio
sectors = {symbol: get_company_sector(symbol) for symbol in SYMBOLS}
sector_exposure = calculate_sector_exposure(PORTFOLIO, latest_prices, sectors)
print(sectors)
sector_exposure

OK  sector_exposure toy Technology: 0.500000  (expected 0.500000)
OK  sector_exposure toy Healthcare: 0.500000  (expected 0.500000)
OK  sector_exposure vs reference Technology: 0.500000  (expected 0.500000)
OK  sector_exposure vs reference Healthcare: 0.500000  (expected 0.500000)
[NVDA] Alpha Vantage OVERVIEW had no Sector field. Using fallback sector instead.
[MSFT] Alpha Vantage OVERVIEW had no Sector field. Using fallback sector instead.
[TSLA] Alpha Vantage OVERVIEW had no Sector field. Using fallback sector instead.
{'AAPL': 'TECHNOLOGY', 'NVDA': 'Technology', 'MSFT': 'Technology', 'GOOGL': 'COMMUNICATION SERVICES', 'TSLA': 'Consumer Discretionary'}


{'TECHNOLOGY': np.float64(0.18492068015009133),
 'Technology': np.float64(0.4386200235754971),
 'COMMUNICATION SERVICES': np.float64(0.25736524323854143),
 'Consumer Discretionary': np.float64(0.11909405303587016)}

## Step 8 — `calculate_correlation_matrix()`

Pairwise return correlation across the 5 holdings.

`calculate_correlation_matrix  `

Purpose:
Checks whether your stocks tend to move together.

Explanation:
Helps to answer Is my portfolio actually diversified? If everything moves together, diversification is weaker.

In [32]:
from risklensaidev.risk import calculate_correlation_matrix

` _pearson_correlation_reference `

Purpose:
Calculates correlation manually.

Explanation:
To check the real correlation calculation.

In [33]:


def _pearson_correlation_reference(x: list, y: list) -> float:
    """Independent plain-Python Pearson correlation between two series,
    used only to cross-check individual entries of calculate_correlation_matrix()."""
    n = len(x)
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    covariance = sum((x[i] - mean_x) * (y[i] - mean_y) for i in range(n))
    std_x = sum((xi - mean_x) ** 2 for xi in x) ** 0.5
    std_y = sum((yi - mean_y) ** 2 for yi in y) ** 0.5
    return covariance / (std_x * std_y)



In [34]:

# --- Validation: hand-calculated toy example ---
# X is an arbitrary return series. Y = 2*X (perfectly, positively linearly
# related -> correlation must be exactly +1, regardless of X's actual values).
# Z = -X (perfectly, negatively linearly related -> correlation must be -1).
toy_x = [0.01, 0.02, -0.01, 0.03]
toy_y = [2 * v for v in toy_x]
toy_z = [-v for v in toy_x]
toy_corr_returns = {"X": pd.Series(toy_x), "Y": pd.Series(toy_y), "Z": pd.Series(toy_z)}

toy_corr_matrix = calculate_correlation_matrix(toy_corr_returns)
assert_close(toy_corr_matrix.loc["X", "X"], 1.0, label="correlation toy X-X (diagonal)")
assert_close(toy_corr_matrix.loc["X", "Y"], 1.0, label="correlation toy X-Y (scaled copy)")
assert_close(toy_corr_matrix.loc["X", "Z"], -1.0, label="correlation toy X-Z (negated copy)")

# --- Validation: cross-check against the independent reference implementation ---
reference_xy_corr = _pearson_correlation_reference(toy_x, toy_y)
assert_close(toy_corr_matrix.loc["X", "Y"], reference_xy_corr, label="correlation vs reference X-Y")

# Apply to the real portfolio
correlation_matrix = calculate_correlation_matrix(returns)
correlation_matrix

OK  correlation toy X-X (diagonal): 1.000000  (expected 1.000000)
OK  correlation toy X-Y (scaled copy): 1.000000  (expected 1.000000)
OK  correlation toy X-Z (negated copy): -1.000000  (expected -1.000000)
OK  correlation vs reference X-Y: 1.000000  (expected 1.000000)


,AAPL,NVDA,MSFT,GOOGL,TSLA
AAPL,1.000000,0.027874,0.096142,0.029470,0.171795
NVDA,0.027874,1.000000,0.243052,0.274858,0.424404
MSFT,0.096142,0.243052,1.000000,0.186664,0.183645
GOOGL,0.029470,0.274858,0.186664,1.000000,0.421468
TSLA,0.171795,0.424404,0.183645,0.421468,1.000000


## Step 9 — `calculate_contribution_to_loss()`

Each holding's share of total portfolio loss over the period — this later ranks News targets in the full-investigation route.

` calculate_contribution_to_loss `

Purpose:
Finds which stock caused the biggest part of the portfolio's loss.

Explanation:
Helps to answer "What is driving my portfolio loss?" It is used later for the ai flow to decide which company should be investigated for news


In [35]:
from risklensaidev.risk import calculate_contribution_to_loss

`_calculate_contribution_to_loss_reference`

Purpose:
Calculates loss contribution another way.

Explanation:
To check the real loss-contribution function.

In [36]:
def _calculate_contribution_to_loss_reference(portfolio: list, start_prices: dict, end_prices: dict) -> dict:
    """Independent plain-Python reimplementation, used only to cross-check
    calculate_contribution_to_loss() above (see Step 0 methodology)."""
    symbols, changes = [], []
    for h in portfolio:
        symbols.append(h["symbol"])
        changes.append(h["quantity"] * (end_prices[h["symbol"]] - start_prices[h["symbol"]]))
    total = sum(changes)
    return {s: c / total for s, c in zip(symbols, changes)}



In [37]:



# --- Validation: hand-calculated toy example ---
# Reuse the Step 6 toy portfolio (A qty=10, B qty=5, C qty=2).
# Start prices: A=10, B=20, C=100. End prices: A=8, B=15, C=90 (all three lose value).
#   A: 10 * (8  - 10)  = -20
#   B: 5  * (15 - 20)  = -25
#   C: 2  * (90 - 100) = -20
#   total change = -65
#   contributions: A = -20/-65 = 4/13 = 0.3077, B = -25/-65 = 5/13 = 0.3846, C = 4/13 = 0.3077
#   (B is the biggest loss driver -- and 4/13 + 5/13 + 4/13 = 1.0 exactly)
toy_start_prices = {"A": 10, "B": 20, "C": 100}
toy_end_prices = {"A": 8, "B": 15, "C": 90}

toy_contribution = calculate_contribution_to_loss(toy_portfolio, toy_start_prices, toy_end_prices)
assert_close(toy_contribution["A"], 4 / 13, label="contribution_to_loss toy A")
assert_close(toy_contribution["B"], 5 / 13, label="contribution_to_loss toy B")
assert_close(toy_contribution["C"], 4 / 13, label="contribution_to_loss toy C")

# --- Validation: cross-check against the independent reference implementation ---
reference_contribution = _calculate_contribution_to_loss_reference(toy_portfolio, toy_start_prices, toy_end_prices)
for symbol in reference_contribution:
    assert_close(toy_contribution[symbol], reference_contribution[symbol], label=f"contribution_to_loss vs reference {symbol}")

# Apply to the real portfolio: start = first close in the fetched window, end = latest close
start_prices = {symbol: df["close"].iloc[0] for symbol, df in prices.items()}
contribution_to_loss = calculate_contribution_to_loss(PORTFOLIO, start_prices, latest_prices)
contribution_to_loss

OK  contribution_to_loss toy A: 0.307692  (expected 0.307692)
OK  contribution_to_loss toy B: 0.384615  (expected 0.384615)
OK  contribution_to_loss toy C: 0.307692  (expected 0.307692)
OK  contribution_to_loss vs reference A: 0.307692  (expected 0.307692)
OK  contribution_to_loss vs reference B: 0.384615  (expected 0.384615)
OK  contribution_to_loss vs reference C: 0.307692  (expected 0.307692)


{'AAPL': np.float64(0.2556582806191559),
 'NVDA': np.float64(0.24428253922329335),
 'MSFT': np.float64(0.4007417246413024),
 'GOOGL': np.float64(0.2621286445113691),
 'TSLA': np.float64(-0.16281118899512087)}

## Step 10 — Full pipeline run

Execute all seven calculations together across the real portfolio for a chosen date range, assemble the combined `risk_results` JSON exactly as it would be persisted to `INVESTIGATION.risk_results`, and sanity-check the numbers agree with each other as a whole.

In [ ]:
import json

from risklensaidev.risk import run_full_investigation

risk_results = run_full_investigation(PORTFOLIO, prices, sectors)

# --- Sanity checks: do the pieces agree with each other as a whole? ---
# These sums are mathematical invariants of how each function is defined
# (weights/exposures/contributions are all normalized to a total), so if any
# of these fail, it means a bug crept into one of Steps 6/7/9 -- not that the
# underlying math is somehow allowed to not add up.
assert_close(sum(risk_results["concentration"]["weights"].values()), 1.0, label="sanity: concentration weights sum to 1.0")
assert_close(sum(risk_results["sector_exposure"].values()), 1.0, label="sanity: sector_exposure sums to 1.0")
assert_close(sum(risk_results["loss_contribution"].values()), 1.0, label="sanity: loss_contribution sums to 1.0")
for s in SYMBOLS:
    assert_close(risk_results["correlation_matrix"][s][s], 1.0, label=f"sanity: correlation diagonal {s}")

top_loss_driver = max(risk_results["loss_contribution"], key=risk_results["loss_contribution"].get)
print(f"\nLargest holding by portfolio weight : {risk_results['concentration']['largest_holding_symbol']}")
print(f"Largest loss-contribution driver     : {top_loss_driver} ({risk_results['loss_contribution'][top_loss_driver]:.1%})")
print("^ this is the symbol that would become the News target in a full investigation (top-1 by contribution_to_loss, per CLAUDE.md).")

# Assemble the full record exactly as it would be persisted (INVESTIGATION table shape)
investigation = {
    "portfolio_id": 1,
    "query": "Investigate my portfolio risk",
    "intent": "full_risk_investigation",
    "status": "completed",
    "start_date": str(prices[SYMBOLS[0]].index[0].date()),
    "end_date": str(prices[SYMBOLS[0]].index[-1].date()),
    "risk_results": risk_results,
}

print("\n--- INVESTIGATION.risk_results (as it would be persisted) ---")
print(json.dumps(investigation, indent=2))

OK  sanity: concentration weights sum to 1.0: 1.000000  (expected 1.000000)
OK  sanity: sector_exposure sums to 1.0: 1.000000  (expected 1.000000)
OK  sanity: loss_contribution sums to 1.0: 1.000000  (expected 1.000000)
OK  sanity: correlation diagonal AAPL: 1.000000  (expected 1.000000)
OK  sanity: correlation diagonal NVDA: 1.000000  (expected 1.000000)
OK  sanity: correlation diagonal MSFT: 1.000000  (expected 1.000000)
OK  sanity: correlation diagonal GOOGL: 1.000000  (expected 1.000000)
OK  sanity: correlation diagonal TSLA: 1.000000  (expected 1.000000)

Largest holding by portfolio weight : GOOGL
Largest loss-contribution driver     : MSFT (40.1%)
^ this is the symbol that would become the News target in a full investigation (top-1 by contribution_to_loss, per CLAUDE.md).

--- INVESTIGATION.risk_results (as it would be persisted) ---
{
  "portfolio_id": 1,
  "query": "Investigate my portfolio risk",
  "intent": "full_risk_investigation",
  "status": "completed",
  "start_date": 

: 